In [2]:
import sys
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project")  # adjust if your notebook is nested differently
from functions_v2 import *

In [5]:
import numpy as np
import networkx as nx
from sksparse.cholmod import cholesky as sparse_cholesky

In [12]:
# 1. Load
edges, n_vertices, edge_weights = load_graph(r"../../data/raw/bio-grid-yeast.edges")
print(f"Raw: n_vertices={n_vertices}, edges={len(edges)}")

# 2. Remap if needed (load_graph should now handle this automatically, but confirm)
all_nodes = sorted(set(v for e in edges for v in e))
print(f"min: {min(all_nodes)}, max: {max(all_nodes)}, count: {len(all_nodes)}")
# should already be contiguous if load_graph's remapping fix is in place

# 3. Confirm connectivity
G_nx = nx.Graph()
G_nx.add_edges_from(edges)
print("Components:", nx.number_connected_components(G_nx))

# 4. Phase 1
A, D, L = build_graph_matrices(edges, n_vertices)

# 5. Phase 2
lambda_min = compute_lambda_min(L, D)
L_sigma = build_shifted_laplacian(L, D, lambda_min)
print("L_sigma type:", type(L_sigma))

✓ Loaded: ../../data/raw/bio-grid-yeast.edges
  Vertices : 6008
  Edges    : 156945
  Weighted : no

Raw: n_vertices=6008, edges=156945
min: 0, max: 6007, count: 6008
Components: 1
✓ Phase 1 complete: A, D, L built as sparse matrices
  Matrix size : 6008 x 6008
  Degree range: [1, 2557]
  Non-zeros in L: 319898

✓ lambda_min = 0.096955
  (eigenvalues found: [0.         0.09695516])
✓ Phase 2 complete: L_sigma built (sigma^2 = 0.25)
  L_sigma type: sparse

L_sigma type: <class 'scipy.sparse._csr.csr_matrix'>


In [14]:
from networkx.algorithms.community import greedy_modularity_communities, modularity

communities = list(greedy_modularity_communities(G_nx))
communities_sorted = sorted(communities, key=len, reverse=True)

print(f"Found {len(communities_sorted)} communities")
print(f"Sizes (top 10): {[len(c) for c in communities_sorted[:10]]}")

mod_score = modularity(G_nx, communities_sorted)
print(f"Modularity: {mod_score:.4f}")

Found 22 communities
Sizes (top 10): [2686, 1867, 1172, 115, 79, 16, 13, 11, 7, 5]
Modularity: 0.2517


In [16]:
boundary_fraction = (2686 + 1867) / 6008
print(f"Boundary fraction if using top 2 communities: {boundary_fraction:.4f}")

Boundary fraction if using top 2 communities: 0.7578


In [21]:
def find_diameter_endpoints(G_nx, n_sample=5, k_hop=4):
    best_dist = 0
    best_v, best_w = None, None
    start_nodes = list(G_nx.nodes())[:n_sample]

    for start in start_nodes:
        lengths = nx.single_source_shortest_path_length(G_nx, start)
        v = max(lengths, key=lengths.get)
        lengths_v = nx.single_source_shortest_path_length(G_nx, v)
        w = max(lengths_v, key=lengths_v.get)
        dist = lengths_v[w]
        if dist > best_dist:
            best_dist = dist
            best_v, best_w = v, w

    gamma_in  = set(nx.single_source_shortest_path_length(G_nx, best_v, cutoff=k_hop).keys())
    gamma_out = set(nx.single_source_shortest_path_length(G_nx, best_w, cutoff=k_hop).keys())
    gamma_in  = gamma_in - gamma_out
    gamma_out = gamma_out - gamma_in

    return list(map(int, gamma_in)), list(map(int, gamma_out)), best_dist

gamma_in, gamma_out, diameter = find_diameter_endpoints(G_nx, n_sample=5, k_hop=4)

print(f"Diameter (approx): {diameter} hops")
print(f"gamma_in: {len(gamma_in)} vertices")
print(f"gamma_out: {len(gamma_out)} vertices")

overlap = set(gamma_in) & set(gamma_out)
print(f"Overlap (should be empty): {overlap}")

max_id = max(max(gamma_in), max(gamma_out))
assert max_id < n_vertices

boundary_fraction = (len(gamma_in) + len(gamma_out)) / n_vertices
print(f"Boundary fraction of total graph: {boundary_fraction:.4f}")

Diameter (approx): 5 hops
gamma_in: 43 vertices
gamma_out: 5957 vertices
Overlap (should be empty): set()
Boundary fraction of total graph: 0.9987


In [23]:
gamma_in, gamma_out, diameter = find_diameter_endpoints(G_nx, n_sample=5, k_hop=4)

print(f"Diameter (approx): {diameter} hops")
print(f"gamma_in: {len(gamma_in)} vertices")
print(f"gamma_out: {len(gamma_out)} vertices")

overlap = set(gamma_in) & set(gamma_out)
print(f"Overlap (should be empty): {overlap}")

max_id = max(max(gamma_in), max(gamma_out))
assert max_id < n_vertices

boundary_fraction = (len(gamma_in) + len(gamma_out)) / n_vertices
print(f"Boundary fraction of total graph: {boundary_fraction:.4f}")

Diameter (approx): 5 hops
gamma_in: 43 vertices
gamma_out: 5957 vertices
Overlap (should be empty): set()
Boundary fraction of total graph: 0.9987


In [25]:
gamma_in, gamma_out, diameter = find_diameter_endpoints(G_nx, n_sample=5, k_hop=1)

print(f"Diameter (approx): {diameter} hops")
print(f"gamma_in: {len(gamma_in)} vertices")
print(f"gamma_out: {len(gamma_out)} vertices")

boundary_fraction = (len(gamma_in) + len(gamma_out)) / n_vertices
print(f"Boundary fraction: {boundary_fraction:.4f}")

Diameter (approx): 5 hops
gamma_in: 2 vertices
gamma_out: 2 vertices
Boundary fraction: 0.0007


In [30]:
def build_capped_aggregation_v2(G_nx, gamma_in, gamma_out, max_size=10):
    gamma_in_set = set(gamma_in)
    gamma_out_set = set(gamma_out)
    boundary_set = gamma_in_set | gamma_out_set

    aggregate_of = {}
    agg_sizes = {}
    agg_has_gamma_in = {}
    agg_has_gamma_out = {}
    next_agg_id = 0

    def forbidden_pair(a, b):
        return (a in gamma_in_set and b in gamma_out_set) or \
               (a in gamma_out_set and b in gamma_in_set)

    # STEP 1: process boundary vertices FIRST, forcing them to pair with an
    # interior neighbor specifically (never with each other, never alone)
    boundary_nodes_by_degree = sorted(boundary_set, key=lambda v: G_nx.degree(v))

    for v in boundary_nodes_by_degree:
        if v in aggregate_of:
            continue
        partner = None
        for neighbor in G_nx.neighbors(v):
            if neighbor not in aggregate_of and neighbor not in boundary_set:
                partner = neighbor
                break
        if partner is not None:
            aggregate_of[v] = next_agg_id
            aggregate_of[partner] = next_agg_id
            agg_sizes[next_agg_id] = 2
            agg_has_gamma_in[next_agg_id] = v in gamma_in_set
            agg_has_gamma_out[next_agg_id] = v in gamma_out_set
            next_agg_id += 1

    # STEP 2: process everyone else (interior vertices, plus any leftover
    # unaggregated boundary vertices)
    remaining_by_degree = sorted(
        [v for v in G_nx.nodes() if v not in aggregate_of],
        key=lambda v: G_nx.degree(v)
    )

    for v in remaining_by_degree:
        if v in aggregate_of:
            continue
        partner = None
        for neighbor in G_nx.neighbors(v):
            if neighbor not in aggregate_of and not forbidden_pair(v, neighbor):
                partner = neighbor
                break
        if partner is not None:
            aggregate_of[v] = next_agg_id
            aggregate_of[partner] = next_agg_id
            agg_sizes[next_agg_id] = 2
            agg_has_gamma_in[next_agg_id] = (v in gamma_in_set) or (partner in gamma_in_set)
            agg_has_gamma_out[next_agg_id] = (v in gamma_out_set) or (partner in gamma_out_set)
            next_agg_id += 1
        else:
            joined = False
            for neighbor in G_nx.neighbors(v):
                if neighbor in aggregate_of and not forbidden_pair(v, neighbor):
                    agg_id = aggregate_of[neighbor]
                    v_is_in = v in gamma_in_set
                    v_is_out = v in gamma_out_set
                    if (v_is_in and agg_has_gamma_out.get(agg_id, False)) or \
                       (v_is_out and agg_has_gamma_in.get(agg_id, False)):
                        continue
                    if agg_sizes[agg_id] < max_size:
                        aggregate_of[v] = agg_id
                        agg_sizes[agg_id] += 1
                        agg_has_gamma_in[agg_id] = agg_has_gamma_in.get(agg_id, False) or v_is_in
                        agg_has_gamma_out[agg_id] = agg_has_gamma_out.get(agg_id, False) or v_is_out
                        joined = True
                        break
            if not joined:
                aggregate_of[v] = next_agg_id
                agg_sizes[next_agg_id] = 1
                agg_has_gamma_in[next_agg_id] = v in gamma_in_set
                agg_has_gamma_out[next_agg_id] = v in gamma_out_set
                next_agg_id += 1

    unique_ids = sorted(set(aggregate_of.values()))
    relabel = {old: new for new, old in enumerate(unique_ids)}
    aggregate_of = {v: relabel[a] for v, a in aggregate_of.items()}
    return aggregate_of, len(unique_ids)

In [32]:
aggregate_of, n_coarse = build_capped_aggregation_v2(G_nx, gamma_in, gamma_out, max_size=10)

groups = {}
for v, a in aggregate_of.items():
    groups.setdefault(a, []).append(v)
sizes = [len(m) for m in groups.values()]

print(f"n_coarse: {n_coarse} (from {n_vertices} fine vertices, {n_vertices/n_coarse:.1f}x reduction)")
print(f"Size distribution — min: {min(sizes)}, max: {max(sizes)}, mean: {np.mean(sizes):.2f}")
print(f"Singletons: {sizes.count(1)} ({sizes.count(1)/n_coarse*100:.1f}%)")

gamma_in_coarse = sorted(set(aggregate_of[v] for v in gamma_in))
gamma_out_coarse = sorted(set(aggregate_of[v] for v in gamma_out))
overlap = set(gamma_in_coarse) & set(gamma_out_coarse)
print(f"Overlap: {overlap}")

boundary_coarse = set(gamma_in_coarse) | set(gamma_out_coarse)
interior_coarse = [v for v in range(n_coarse) if v not in boundary_coarse]
print(f"Interior coarse vertices: {len(interior_coarse)} ({len(interior_coarse)/n_coarse*100:.1f}%)")

n_coarse: 2918 (from 6008 fine vertices, 2.1x reduction)
Size distribution — min: 1, max: 10, mean: 2.06
Singletons: 122 (4.2%)
Overlap: set()
Interior coarse vertices: 2916 (99.9%)


In [37]:
def build_coarse_graph_edges(edges, aggregate_of):
    coarse_contribs = {}
    for idx, (i, j) in enumerate(edges):
        ai, aj = aggregate_of[i], aggregate_of[j]
        if ai == aj:
            continue
        key = tuple(sorted([ai, aj]))
        coarse_contribs.setdefault(key, []).append(idx)
    coarse_edges = list(coarse_contribs.keys())
    return coarse_edges, coarse_contribs

coarse_edges, coarse_contribs = build_coarse_graph_edges(edges, aggregate_of)
print(f"coarse edges: {len(coarse_edges)} (from {len(edges)} fine edges)")

coarse edges: 145197 (from 156945 fine edges)


In [38]:
coarse_edges, coarse_contribs = build_coarse_graph_edges(edges, aggregate_of)
print(f"coarse edges: {len(coarse_edges)} (from {len(edges)} fine edges)")

coarse_edges_arr = np.array(coarse_edges)
i_arr_c = coarse_edges_arr[:, 0]
j_arr_c = coarse_edges_arr[:, 1]
row_idx_c = np.concatenate([i_arr_c, j_arr_c, i_arr_c, j_arr_c])
col_idx_c = np.concatenate([i_arr_c, j_arr_c, j_arr_c, i_arr_c])

gamma_in_coarse_set = set(gamma_in_coarse)
gamma_out_coarse_set = set(gamma_out_coarse)
boundary_coarse = set(gamma_in_coarse) | set(gamma_out_coarse)
interior_coarse = np.array([v for v in range(n_coarse) if v not in boundary_coarse])

p_known_coarse = np.full(n_coarse, np.nan)
for v in gamma_in_coarse:
    p_known_coarse[v] = 1.0
for v in gamma_out_coarse:
    p_known_coarse[v] = 0.0

# fine graph setup
B = build_incidence_matrix(edges, n_vertices)
factor = sparse_cholesky(L_sigma)
edges_arr = np.array(edges)
i_arr = edges_arr[:, 0]
j_arr = edges_arr[:, 1]
row_idx = np.concatenate([i_arr, j_arr, i_arr, j_arr])
col_idx = np.concatenate([i_arr, j_arr, j_arr, i_arr])
boundary = set(gamma_in) | set(gamma_out)
interior = np.array([v for v in range(n_vertices) if v not in boundary])
p_known = np.full(n_vertices, np.nan)
for v in gamma_in:
    p_known[v] = 1.0
for v in gamma_out:
    p_known[v] = 0.0
gamma_out_set = set(gamma_out)

print(f"Fine: boundary={len(boundary)}, interior={len(interior)}")
print(f"Coarse: boundary={len(boundary_coarse)}, interior={len(interior_coarse)}")

coarse edges: 145197 (from 156945 fine edges)


C:\Users\Rakesh\AppData\Local\Temp\ipykernel_6024\4097140868.py:23: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = sparse_cholesky(L_sigma)


Fine: boundary=4, interior=6004
Coarse: boundary=2, interior=2916


In [41]:
row_idx_c = np.concatenate([i_arr_c, j_arr_c, i_arr_c, j_arr_c])
col_idx_c = np.concatenate([i_arr_c, j_arr_c, j_arr_c, i_arr_c])

gamma_in_coarse_set = set(gamma_in_coarse)
gamma_out_coarse_set = set(gamma_out_coarse)
boundary_coarse = set(gamma_in_coarse) | set(gamma_out_coarse)
interior_coarse = np.array([v for v in range(n_coarse) if v not in boundary_coarse])

p_known_coarse = np.full(n_coarse, np.nan)
for v in gamma_in_coarse:
    p_known_coarse[v] = 1.0
for v in gamma_out_coarse:
    p_known_coarse[v] = 0.0

# fine graph setup
B = build_incidence_matrix(edges, n_vertices)
factor = sparse_cholesky(L_sigma)
edges_arr = np.array(edges)
i_arr = edges_arr[:, 0]
j_arr = edges_arr[:, 1]
row_idx = np.concatenate([i_arr, j_arr, i_arr, j_arr])
col_idx = np.concatenate([i_arr, j_arr, j_arr, i_arr])
boundary = set(gamma_in) | set(gamma_out)
interior = np.array([v for v in range(n_vertices) if v not in boundary])
p_known = np.full(n_vertices, np.nan)
for v in gamma_in:
    p_known[v] = 1.0
for v in gamma_out:
    p_known[v] = 0.0
gamma_out_set = set(gamma_out)

print(f"Fine: boundary={len(boundary)}, interior={len(interior)}")
print(f"Coarse: boundary={len(boundary_coarse)}, interior={len(interior_coarse)}")

C:\Users\Rakesh\AppData\Local\Temp\ipykernel_6024\349366724.py:17: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = sparse_cholesky(L_sigma)


Fine: boundary=4, interior=6004
Coarse: boundary=2, interior=2916


In [46]:
from scipy.sparse.linalg import spsolve
from scipy.sparse import coo_matrix

def run_one_paired_sample(seed):
    rng = np.random.RandomState(seed)
    w = rng.randn(len(edges))

    f = B @ w
    u = factor.solve_A(np.sqrt(lambda_min) * f)
    k_vals = np.exp((u[i_arr] + u[j_arr]) / 2)

    diag_data = np.concatenate([k_vals, k_vals])
    offdiag_data = np.concatenate([-k_vals, -k_vals])
    data = np.concatenate([diag_data, offdiag_data])
    L_k = coo_matrix((data, (row_idx, col_idx)), shape=(n_vertices, n_vertices)).tocsr()

    L_interior = L_k[interior, :][:, interior]
    rhs = -L_k[interior, :][:, list(boundary)] @ p_known[list(boundary)]
    p_interior = spsolve(L_interior, rhs)

    p = np.zeros(n_vertices)
    p[interior] = p_interior
    for v in gamma_in:
        p[v] = 1.0
    for v in gamma_out:
        p[v] = 0.0

    p_diff = np.abs(p[i_arr] - p[j_arr])
    outlet_mask = np.array([i in gamma_out_set or j in gamma_out_set for i, j in edges])
    Q_fine = np.sum(k_vals[outlet_mask] * p_diff[outlet_mask])

    coarse_k_vals = np.array([sum(k_vals[idx] for idx in coarse_contribs[e]) for e in coarse_edges])

    diag_c = np.concatenate([coarse_k_vals, coarse_k_vals])
    offdiag_c = np.concatenate([-coarse_k_vals, -coarse_k_vals])
    data_c = np.concatenate([diag_c, offdiag_c])
    L_k_c = coo_matrix((data_c, (row_idx_c, col_idx_c)), shape=(n_coarse, n_coarse)).tocsr()

    L_interior_c = L_k_c[interior_coarse, :][:, interior_coarse]
    rhs_c = -L_k_c[interior_coarse, :][:, list(boundary_coarse)] @ p_known_coarse[list(boundary_coarse)]
    p_interior_c = spsolve(L_interior_c, rhs_c)

    p_c = np.zeros(n_coarse)
    p_c[interior_coarse] = p_interior_c
    for v in gamma_in_coarse:
        p_c[v] = 1.0
    for v in gamma_out_coarse:
        p_c[v] = 0.0

    p_diff_c = np.abs(p_c[i_arr_c] - p_c[j_arr_c])
    outlet_mask_c = np.array([i in gamma_out_coarse_set or j in gamma_out_coarse_set for i, j in coarse_edges])
    Q_coarse = np.sum(coarse_k_vals[outlet_mask_c] * p_diff_c[outlet_mask_c])

    return Q_fine, Q_coarse

Q_fine: 32.5392
Q_coarse: 82.7935


In [43]:
qf, qc = run_one_paired_sample(seed=0)
print(f"Q_fine: {qf:.4f}")
print(f"Q_coarse: {qc:.4f}")

NameError: name 'run_one_paired_sample' is not defined

In [48]:
outlet_edges_fine = [(i,j) for i,j in edges if i in gamma_out_set or j in gamma_out_set]
outlet_edges_coarse = [(i,j) for i,j in coarse_edges if i in gamma_out_coarse_set or j in gamma_out_coarse_set]

print(f"Fine outlet edges: {len(outlet_edges_fine)}")
print(f"Coarse outlet edges: {len(outlet_edges_coarse)}")

qf_norm = qf / len(outlet_edges_fine)
qc_norm = qc / len(outlet_edges_coarse)
print(f"Normalized Q_fine: {qf_norm:.6f}")
print(f"Normalized Q_coarse: {qc_norm:.6f}")

Fine outlet edges: 75
Coarse outlet edges: 95
Normalized Q_fine: 0.433855
Normalized Q_coarse: 0.871510


In [50]:
N = 300
Q_fine_samples = np.zeros(N)
Q_coarse_samples = np.zeros(N)

for n in range(N):
    qf, qc = run_one_paired_sample(seed=n)
    Q_fine_samples[n] = qf
    Q_coarse_samples[n] = qc

diff_samples = Q_fine_samples - Q_coarse_samples
correlation = np.corrcoef(Q_fine_samples, Q_coarse_samples)[0, 1]

print(f"\nN = {N} paired samples\n")
print(f"Q_fine   : mean={Q_fine_samples.mean():.6f}  var={Q_fine_samples.var():.6f}")
print(f"Q_coarse : mean={Q_coarse_samples.mean():.6f}  var={Q_coarse_samples.var():.6f}")
print(f"Q_fine - Q_coarse : mean={diff_samples.mean():.6f}  var={diff_samples.var():.6f}")
print(f"\nCorrelation(Q_fine, Q_coarse): {correlation:.4f}")
print(f"Variance reduction (var(Q_fine)/var(diff)): {Q_fine_samples.var()/diff_samples.var():.2f}x")

Exception ignored in: 'zmq.backend.cython.message.Frame.__dealloc__'
Traceback (most recent call last):
  File "zmq\\backend\\cython\\checkrc.pxd", line 13, in zmq.backend.cython.checkrc._check_rc
KeyboardInterrupt: 


KeyboardInterrupt: 